# Preprocesamiento para el modelo

Ingeniería de características sobre las variables seleccionadas en el análisis exploratorio (`MODEL_FEATURES` en `pipeline_config.yaml`). El resultado es un pipeline de scikit-learn construido con transformadores de `feature-engine` y los datasets procesados que sirven de input para el notebook de modelado.

| Paso | Técnica | Variables |
|---|---|---|
| Imputación numérica | Mediana (`MeanMedianImputer`) | Continuas y discretas |
| Imputación categórica | Moda (`CategoricalImputer`) | Categóricas |
| Codificación | Ordinal ordenada por la media del target (`OrdinalEncoder`) | Categóricas |
| Outliers | Recorte por IQR (`Winsorizer`) | Continuas |
| Transformación | Yeo-Johnson (`YeoJohnsonTransformer`) | Continuas con \|skew\| > 0.75 |
| Escalado | `StandardScaler` | Todas |

El target se modela como `log1p(SalePrice)`. Todos los transformadores se ajustan únicamente con el split de entrenamiento para evitar fuga de información.

## 1. Librerías

Carga de librerías de manipulación de datos, transformadores de `feature-engine`, utilidades de scikit-learn y rutas del proyecto.

In [1]:
# Librerías
import joblib
import numpy as np
import pandas as pd
import yaml
from scipy.stats import skew
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from feature_engine.encoding import OrdinalEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper

from house_pricing_a.config import MODELS_DIR, PROCESSED_DATA_DIR, PROJ_ROOT, RAW_DATA_DIR

pd.set_option("display.max_columns", None)

2026-11-03 19:06:13.191 | INFO     | house_pricing_a.config:<module>:11 - PROJ_ROOT path is: /Users/prengbiba/Dropbox/Compu/Clases/Postgrado/PAPD/L/Data Product Development/2026/house_pricing_A


## 2. Carga de datos y configuración

Lectura de los datasets de entrenamiento y prueba, y de la lista de features definida en `pipeline_config.yaml`. La columna `Id` se usa como índice para conservar la trazabilidad de cada registro a lo largo del pipeline.

In [2]:
# Rutas y parámetros: ajustar si cambia la fuente
DATA_DIR = RAW_DATA_DIR
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"
PIPELINE_CONFIG_PATH = PROJ_ROOT / "pipeline_config.yaml"

TARGET = "SalePrice"
ID_COL = "Id"

with open(PIPELINE_CONFIG_PATH) as f:
    pipeline_config = yaml.safe_load(f)
MODEL_FEATURES = pipeline_config["MODEL_FEATURES"]

train = pd.read_csv(DATA_DIR / TRAIN_FILE, index_col=ID_COL)
test = pd.read_csv(DATA_DIR / TEST_FILE, index_col=ID_COL)

print(f"train: {train.shape} | test: {test.shape} | features: {len(MODEL_FEATURES)}")

train: (1460, 80) | test: (1459, 79) | features: 30


## 3. Clasificación de las features

Agrupación de `MODEL_FEATURES` en continuas, discretas y categóricas con el mismo criterio del análisis exploratorio: una variable numérica es discreta si tiene hasta `MAX_UNIQUE_DISCRETE` valores únicos. Cada paso del pipeline se aplica sobre el grupo que le corresponde.

In [3]:
# Clasificación de features por tipo
MAX_UNIQUE_DISCRETE = 15

numericas = train[MODEL_FEATURES].select_dtypes(include="number").columns.tolist()
categoricas = train[MODEL_FEATURES].select_dtypes(exclude="number").columns.tolist()

n_unique = train[numericas].nunique()
discretas = [col for col in numericas if n_unique[col] <= MAX_UNIQUE_DISCRETE]
continuas = [col for col in numericas if n_unique[col] > MAX_UNIQUE_DISCRETE]

print(f"Continuas ({len(continuas)}): {continuas}")
print(f"Discretas ({len(discretas)}): {discretas}")
print(f"Categóricas ({len(categoricas)}): {categoricas}")

Continuas (10): ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']
Discretas (10): ['OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'MSSubClass', 'Fireplaces', 'OverallCond', 'HalfBath', 'BedroomAbvGr', 'BsmtFullBath']
Categóricas (10): ['Neighborhood', 'ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'FireplaceQu', 'Foundation', 'GarageType', 'BsmtFinType1', 'HeatingQC']


## 4. División train/validación y transformación del target

División 80/20 de `train.csv` en entrenamiento y validación. El pipeline se ajusta solo con entrenamiento y se aplica a validación y a `test.csv`.

El target se transforma con `log1p` para reducir su asimetría; las predicciones se revierten a la escala original con `expm1`.

In [4]:
# Split train/validación y target en escala logarítmica
TEST_SIZE = 0.2
RANDOM_STATE = 42
TARGET_LOG = f"{TARGET}_log"

X_train, X_val, y_train, y_val = train_test_split(
    train[MODEL_FEATURES],
    np.log1p(train[TARGET]).rename(TARGET_LOG),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
X_test = test[MODEL_FEATURES]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Asimetría del target: original = {skew(train[TARGET]):.2f} | log1p = {skew(y_train):.2f}")

X_train: (1168, 30) | X_val: (292, 30) | X_test: (1459, 30)
Asimetría del target: original = 1.88 | log1p = 0.12


## 5. Imputación de variables numéricas y categóricas

- **Numéricas:** se reemplazan los faltantes por la mediana, robusta ante el sesgo de las variables.
- **Categóricas:** se reemplazan los faltantes por la moda (categoría más frecuente).

Los imputadores se aplican a todas las variables de cada grupo, no solo a las que tienen faltantes en entrenamiento, ya que `test.csv` presenta faltantes en columnas que en entrenamiento están completas (p. ej. `GarageCars`, `TotalBsmtSF`, `KitchenQual`).

In [5]:
# Imputación: mediana para numéricas, moda para categóricas
num_imputer = MeanMedianImputer(imputation_method="median", variables=numericas)
cat_imputer = CategoricalImputer(imputation_method="frequent", variables=categoricas)

X_train_imp = num_imputer.fit_transform(X_train)
X_train_imp = cat_imputer.fit_transform(X_train_imp)

faltantes = pd.DataFrame({
    "faltantes_antes": X_train.isna().sum(),
    "faltantes_despues": X_train_imp.isna().sum(),
    "valor_imputado": pd.Series({**num_imputer.imputer_dict_, **cat_imputer.imputer_dict_}),
})
faltantes[faltantes["faltantes_antes"] > 0]

,faltantes_antes,faltantes_despues,valor_imputado
GarageYrBlt,64,0,1980.0
MasVnrArea,6,0,0.0
LotFrontage,217,0,70.0
BsmtQual,28,0,TA
GarageFinish,64,0,Unf
FireplaceQu,547,0,Gd
GarageType,64,0,Attchd
BsmtFinType1,28,0,Unf


## 6. Codificación de variables categóricas

Codificación ordinal ordenada por target: cada categoría se reemplaza por un entero según el orden de la media de `SalePrice_log` en entrenamiento (0 = menor precio medio). Genera una sola columna por variable y conserva la relación monótona con el precio.

Las categorías no vistas en entrenamiento se codifican como `-1` (`unseen="encode"`), para que el pipeline no falle con valores nuevos en validación o test.

In [6]:
# Codificación ordinal ordenada por la media del target
encoder = OrdinalEncoder(encoding_method="ordered", variables=categoricas, unseen="encode")
X_train_enc = encoder.fit_transform(X_train_imp, y_train)

# Ejemplo de mapeo: categoría -> código
pd.Series(encoder.encoder_dict_["KitchenQual"], name="codigo").rename_axis("KitchenQual").to_frame()

,codigo
KitchenQual,
Fa,0
TA,1
Gd,2
Ex,3


## 7. Tratamiento de outliers

Recorte (capping) de valores extremos en las variables continuas con el método IQR: los valores por debajo de Q1 − 1.5·IQR o por encima de Q3 + 1.5·IQR se reemplazan por el límite correspondiente. No se eliminan filas, por lo que el paso es aplicable a validación y test.

In [7]:
# Recorte de outliers por IQR en variables continuas
winsorizer = Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=continuas)
X_train_out = winsorizer.fit_transform(X_train_enc)

limites = pd.DataFrame({
    "limite_inferior": winsorizer.left_tail_caps_,
    "limite_superior": winsorizer.right_tail_caps_,
})
limites["n_recortados"] = (
    (X_train_enc[continuas] < limites["limite_inferior"])
    | (X_train_enc[continuas] > limites["limite_superior"])
).sum()
limites

,limite_inferior,limite_superior,n_recortados
GrLivArea,176.375,2761.375,23
GarageArea,-11.500,928.500,18
TotalBsmtSF,41.125,2054.125,47
1stFlrSF,124.875,2159.875,13
YearBuilt,1881.000,2073.000,5
YearRemodAdd,1909.000,2061.000,0
GarageYrBlt,1903.500,2059.500,1
MasVnrArea,-249.000,415.000,77
BsmtFinSF1,-1081.500,1802.500,5
LotFrontage,31.500,107.500,85


## 8. Transformación de variables continuas

Aplicación de Yeo-Johnson a las variables continuas cuya asimetría, medida después del recorte de outliers, supera `SKEW_THRESHOLD` en valor absoluto. Yeo-Johnson estima la potencia óptima (λ) por variable y admite ceros, presentes en `MasVnrArea` y `BsmtFinSF1`.

In [9]:
# Selección de variables sesgadas y transformación Yeo-Johnson
SKEW_THRESHOLD = 0.75

skew_antes = X_train_out[continuas].apply(skew)
variables_sesgadas = skew_antes[skew_antes.abs() > SKEW_THRESHOLD].index.tolist()

yeo_johnson = YeoJohnsonTransformer(variables=variables_sesgadas)
X_train_yj = yeo_johnson.fit_transform(X_train_out)

pd.DataFrame({
    "skew_antes": skew_antes,
    "skew_despues": X_train_yj[continuas].apply(skew),
    "lambda": pd.Series(yeo_johnson.lambda_dict_),
}).round(2)

,skew_antes,skew_despues,lambda
1stFlrSF,0.61,0.61,NaN
BsmtFinSF1,0.73,0.73,NaN
GarageArea,-0.12,-0.12,NaN
GarageYrBlt,-0.69,-0.69,NaN
GrLivArea,0.59,0.59,NaN
LotFrontage,0.06,0.06,NaN
MasVnrArea,1.27,0.37,-0.22
TotalBsmtSF,0.23,0.23,NaN
YearBuilt,-0.61,-0.61,NaN
YearRemodAdd,-0.51,-0.51,NaN


## 9. Escalado de variables

Estandarización de todas las features a media 0 y desviación estándar 1 con `StandardScaler`, envuelto en `SklearnTransformerWrapper` para conservar el formato DataFrame y los nombres de columna.

In [10]:
# Estandarización de todas las features
scaler = SklearnTransformerWrapper(transformer=StandardScaler(), variables=MODEL_FEATURES)
X_train_scaled = scaler.fit_transform(X_train_yj)

X_train_scaled.describe().loc[["mean", "std"]].round(2)

,GrLivArea,GarageArea,TotalBsmtSF,1stFlrSF,YearBuilt,YearRemodAdd,GarageYrBlt,MasVnrArea,BsmtFinSF1,LotFrontage,OverallQual,GarageCars,FullBath,TotRmsAbvGrd,MSSubClass,Fireplaces,OverallCond,HalfBath,BedroomAbvGr,BsmtFullBath,Neighborhood,ExterQual,BsmtQual,KitchenQual,GarageFinish,FireplaceQu,Foundation,GarageType,BsmtFinType1,HeatingQC
mean,0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0,-0.0,0.0,-0.0,0.0,-0.0,-0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,-0.0,-0.0,0.0,-0.0,0.0,-0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 10. Pipeline de preprocesamiento

Integración de los pasos anteriores en un `Pipeline` de scikit-learn con los mismos parámetros. Se ajusta con el split de entrenamiento y se verifica que su salida coincida con la obtenida paso a paso.

In [11]:
# Pipeline final de preprocesamiento
preprocessing_pipeline = Pipeline([
    ("imputacion_numericas", MeanMedianImputer(imputation_method="median", variables=numericas)),
    ("imputacion_categoricas", CategoricalImputer(imputation_method="frequent", variables=categoricas)),
    ("codificacion", OrdinalEncoder(encoding_method="ordered", variables=categoricas, unseen="encode")),
    ("outliers", Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=continuas)),
    ("transformacion", YeoJohnsonTransformer(variables=variables_sesgadas)),
    ("escalado", SklearnTransformerWrapper(transformer=StandardScaler(), variables=MODEL_FEATURES)),
])
preprocessing_pipeline.fit(X_train, y_train)

pd.testing.assert_frame_equal(preprocessing_pipeline.transform(X_train), X_train_scaled)
print("El pipeline reproduce el resultado paso a paso.")
preprocessing_pipeline

El pipeline reproduce el resultado paso a paso.


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputacion_numericas', ...), ('imputacion_categoricas', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,variables,"['GrLivArea', 'GarageArea', ...]"
,imputation_method,'median'
Name,Type,Value
feature_names_in_,list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"


## 11. Generación de datasets procesados

Aplicación del pipeline a entrenamiento, validación y test, y almacenamiento de los resultados en `data/processed/`, que serán el input del notebook de modelado:

- `train.csv` y `validation.csv`: features procesadas + `SalePrice_log`.
- `test.csv`: features procesadas (sin target).

El pipeline ajustado se guarda en `models/` para aplicarlo a datos nuevos en la etapa de inferencia.

In [12]:
# Transformación de los tres splits y guardado
PIPELINE_PATH = MODELS_DIR / "preprocessing_pipeline.joblib"

train_processed = preprocessing_pipeline.transform(X_train).join(y_train)
val_processed = preprocessing_pipeline.transform(X_val).join(y_val)
test_processed = preprocessing_pipeline.transform(X_test)

datasets = {"train": train_processed, "validation": val_processed, "test": test_processed}
for name, df in datasets.items():
    df.to_csv(PROCESSED_DATA_DIR / f"{name}.csv")
    print(f"{name:<10} {df.shape} | faltantes: {df.isna().sum().sum()} -> {PROCESSED_DATA_DIR / f'{name}.csv'}")

joblib.dump(preprocessing_pipeline, PIPELINE_PATH)
print(f"Pipeline guardado en {PIPELINE_PATH}")

train      (1168, 31) | faltantes: 0 -> /Users/prengbiba/Dropbox/Compu/Clases/Postgrado/PAPD/L/Data Product Development/2026/house_pricing_A/data/processed/train.csv
validation (292, 31) | faltantes: 0 -> /Users/prengbiba/Dropbox/Compu/Clases/Postgrado/PAPD/L/Data Product Development/2026/house_pricing_A/data/processed/validation.csv
test       (1459, 30) | faltantes: 0 -> /Users/prengbiba/Dropbox/Compu/Clases/Postgrado/PAPD/L/Data Product Development/2026/house_pricing_A/data/processed/test.csv
Pipeline guardado en /Users/prengbiba/Dropbox/Compu/Clases/Postgrado/PAPD/L/Data Product Development/2026/house_pricing_A/models/preprocessing_pipeline.joblib
